# 05 — Nấc 5: ACE (Agentic Context Engineering)

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/vinumqa-numerical-reasoning/blob/main/notebooks/05_ace.ipynb)

**Cần GPU.** Pha A ~1.5–2 giờ, pha B ~45 phút.

## Mục đích

Bốn nấc trước đã dùng hết: prompt tĩnh (nấc 2), trọng số (nấc 3), tự soát (nấc 4).
ACE khai thác thứ còn lại: **kinh nghiệm từ tập train, đưa vào prompt lúc chạy**, không đổi
trọng số.

| | Học từ train? | Đổi trọng số? | Cần API ngoài? |
|---|:---:|:---:|:---:|
| Prompt engineering (nấc 2) | ❌ | ❌ | ❌ |
| SFT (nấc 3) | ✅ | ✅ | ❌ |
| Self-eval (nấc 4) | ❌ | ❌ | ❌ |
| **ACE (nấc 5)** | ✅ | ❌ | ❌ |

## Cách hoạt động

```
PHA A (train, có nhãn)                      PHA B (test)
┌────────────────────────────┐              ┌──────────────────────────────┐
│ Generator → program        │              │ truy hồi top-k bullet        │
│ Executor  → PA / EA        │              │          ↓                   │
│ Reflector → chiến lược     │  playbook    │ Bước 1: sinh program         │
│ Verify    → giữ / loại     │ ───────────► │ Bước 2: tự soát & sửa        │
│ Curator   → chèn bullet    │  (đóng băng) │          ↓                   │
└────────────────────────────┘              │ Executor → PA / EA           │
                                            └──────────────────────────────┘
```

Playbook là danh sách bullet chiến lược dạng
*"Khi hỏi tốc độ tăng trưởng giữa hai kỳ, dùng `subtract(gia_tri_moi, gia_tri_cu),
divide(#0, gia_tri_cu)` và giữ dấu âm nếu giảm."* — rút ra từ **chính lỗi của model**, đi
qua quality gate rồi mới được giữ.

## Nguồn gốc và những gì đã phải sửa

Kỹ thuật lấy từ `reference/original_notebooks/02_ace_finqa_ENGLISH.ipynb` (FinQA, tiếng Anh). Tám điểm đã chỉnh cho
ViNumQA, xem `vinumqa/ace/` — quan trọng nhất: bỏ `const_*`, `table_*` đọc theo nhãn hàng,
embedding **đa ngữ** (bản gốc dùng `bge-base-en-v1.5`, xếp hạng bullet tiếng Việt gần như
ngẫu nhiên), và Reflector chạy bằng chính SLM nên **không cần API ngoài**.

## Ba câu hỏi notebook này trả lời

1. ACE cộng thêm bao nhiêu vào nấc trước? (§7)
2. Lợi ích đến từ việc **chọn đúng** bullet hay chỉ vì prompt dài thêm? (§8, nhánh bullet ngẫu nhiên)
3. Bullet nào có công, bullet nào có tội? (§9)

## §1. Môi trường

In [ ]:
# Cài đặt — ghim theo bộ ĐÃ XÁC MINH cài xong sạch trên image Colab hiện tại
# (Python 3.13, torch 2.11.0+cu128, A100).
#
# ⚠ KHÁC bản tham chiếu, và đây là chủ ý:
#   Khối cài đặt gốc ghim transformers==4.56.2 / trl==0.22.2 / xformers==0.0.29.post3.
#   Trên image Colab hiện tại nó THẤT BẠI — nhánh chọn xformers chỉ biết torch 2.8/2.9,
#   gặp torch 2.11 thì rơi vào bản 0.0.29.post3 (dành cho torch 2.5) nên đổ cả khối,
#   mà `%%capture` lại nuốt mất báo lỗi.
#   Bộ dưới đây là bộ pip tự giải ra khi để `unsloth` và `vllm` thoả thuận với nhau.
#   Chênh lệch phiên bản được ghi vào `env` của meta mỗi nấc, nên báo cáo vẫn truy được.
#
# ⏱ 6–12 phút (đã ghim nên pip khỏi dò tìm). Cố ý KHÔNG giấu output để thấy nó còn sống.
import os, time
_t_cai = time.time()
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install unsloth==2026.9.4 transformers==4.57.6 trl==0.24.0 peft==0.20.0 bitsandbytes==0.50.2 xformers==0.0.35
    # vLLM phải khớp CUDA của torch. Bản trên PyPI dựng cho CUDA 13, còn Colab đang
    # CUDA 12.8 → unsloth CHẶN import và báo "No module named 'vllm'" dù gói vẫn có.
    # Wheel dưới đây là bản cu129, đúng cái unsloth khuyến nghị cho hệ CUDA 12.x.
    # Nếu image Colab đổi CUDA: chạy ô này, đọc dòng WARNING của unsloth ở cell sau —
    # nó in ra đúng URL wheel cần dùng, thay vào đây là xong.
    !pip install https://github.com/vllm-project/vllm/releases/download/v0.23.0/vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl
print(f"\n[CÀI ĐẶT] xong sau {(time.time() - _t_cai) / 60:.1f} phút")
print("[CÀI ĐẶT] Colab hiện nút RESTART SESSION thì bấm, rồi chạy lại TỪ CELL #2 "
      "(bỏ qua ô này — cài lại là thừa).")

# Riêng nấc 5 (ACE) cần thêm truy hồi ngữ nghĩa đa ngữ + từ vựng.
# ⚠ Phải GHIM 5.7.0: từ 6.0 trở đi sentence-transformers đòi transformers>=5, mà
#   vLLM 0.23 lại cấm toàn bộ transformers 5.x. Lệnh này chạy sau cùng nên không ghim
#   là nó nâng transformers lên 5.x và làm hỏng vLLM vừa cài xong.
!pip install -q "sentence-transformers==5.7.0" "rank-bm25==0.2.2"

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, sft, stats
from vinumqa.ace import clusters, playbook as pb_mod, reflector as refl_mod
from vinumqa.ace.trainer import AceTrainer
from vinumqa.prompts import PromptKit

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

# Thang prompt LỒNG NHAU: basic ⊂ no_fewshot ⊂ engineered — mỗi nấc thêm đúng một khối
# cắt ra từ prompt hoàn chỉnh, phần chung giống nhau từng ký tự.
LADDER = [
    ("01_basic",           "Nấc 1 — prompt cơ bản (danh sách phép toán + yêu cầu)"),
    ("02_prompt_eng",      "Nấc 2 — prompt hoàn chỉnh (+ hướng dẫn từ khoá + few-shot)"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05c_ace_basic_base", "Nấc 5c — ACE trên prompt cơ bản"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("05c_ace_basic_random_base", "Đối chứng — bullet ngẫu nhiên, prompt cơ bản"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("08_tu_nhat_quan",    "Mới — self-consistency K mẫu (ví dụ cố định)"),
    ("09_vidu_dong",       "Mới — self-consistency + ví dụ truy hồi"),
    ("04_selfeval_base_moi", "Mục tiêu — self-eval + K mẫu + ví dụ truy hồi"),
    ("10_bo_chon",         "Mới — model tự chấm giữa các ứng viên"),
]
LADDER_LABEL = dict(LADDER)

# Những biến phải đi kèm kết quả thì mới truy lại được về sau.
_CFG_KEYS = ("MODEL_NAME", "MODEL_TAG", "TEMPERATURE", "MAX_TOKENS", "REPETITION_PENALTY",
             "MAX_SEQ_LENGTH", "BATCH_SIZE", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "RANDOM_SEED")


def run_env():
    """Môi trường THẬT lúc chạy: commit, GPU, phiên bản thư viện.

    Chỉ đọc thư viện đã nạp (``sys.modules``) chứ không import thêm — vừa nhanh,
    vừa báo đúng những gì thật sự được dùng.
    """
    env = {"python": sys.version.split()[0]}
    try:                                   # bản code nào sinh ra kết quả này
        def _g(*a):
            return subprocess.run(["git", "-C", REPO_DIR, *a],
                                  capture_output=True, text=True).stdout.strip()
        env["commit"] = _g("rev-parse", "--short", "HEAD")
        env["branch"] = _g("rev-parse", "--abbrev-ref", "HEAD")
        env["dirty"] = bool(_g("status", "--porcelain"))
    except Exception:
        pass
    _torch = sys.modules.get("torch")
    if _torch is not None:
        env["torch"] = getattr(_torch, "__version__", "?")
        try:
            if _torch.cuda.is_available():
                _p = _torch.cuda.get_device_properties(0)
                env["gpu"] = _p.name
                env["vram_gb"] = round(_p.total_memory / 1024**3, 1)
                env["cc"] = f"{_p.major}.{_p.minor}"
            else:
                env["gpu"] = "CPU"
        except Exception:
            pass
    else:
        env["gpu"] = "CPU (không nạp torch)"
    for _lib in ("transformers", "trl", "peft", "vllm", "unsloth",
                 "sentence_transformers", "numpy"):
        _m = sys.modules.get(_lib)
        if _m is not None and hasattr(_m, "__version__"):
            env[_lib] = _m.__version__
    return env


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "meta":  f"{stage}_meta.json"}[kind])


# Cấu hình CHUẨN của cả thang bậc — đo trên A100 40GB.
# MAX_SEQ_LENGTH đổi theo GPU (A100 15000 / L4 13500 / T4 8192), mà đổi GPU là đổi
# thành phần lô, đổi kernel, đổi luôn token được lấy mẫu ở temperature 0.1. Hai lần
# chạy khác max_seq KHÔNG so thẳng được, nên phải ghi sang tên nấc khác.
MAX_TOKENS_CHUAN, MAX_SEQ_CHUAN = 4096, 17000



def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: jsonl + meta, kèm một file output thô.

    Chạy với ``MAX_TOKENS`` khác mức chuẩn thì tự ghi sang tên nấc khác. Đổi trần sinh
    là đổi cấu hình, kết quả không so thẳng với thang bậc được — mà nếu cứ ghi đè lên
    tên cũ thì mất luôn bản chuẩn, không lấy lại được nếu không chạy lại GPU.
    """
    _hau_to = ""
    _mt, _ms = globals().get("MAX_TOKENS"), globals().get("MAX_SEQ_LENGTH")
    if _mt and _mt != MAX_TOKENS_CHUAN:
        _hau_to += f"_tok{_mt}"
    if _ms and _ms != MAX_SEQ_CHUAN:
        _hau_to += f"_seq{_ms}"
    if _hau_to and not stage.endswith(_hau_to):
        stage = f"{stage}{_hau_to}"
        if not quiet:
            print(f"[GHI] ⚠ cấu hình khác chuẩn (max_tokens={_mt}, max_seq={_ms}, "
                  f"GPU={globals().get('_GPU', '?')}) → ghi sang nấc '{stage}'.")
            print( "       Kết quả khác GPU/khác trần KHÔNG so thẳng với thang bậc chuẩn.")
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            "enable_thinking": getattr(globals().get("prompt_kit", None),
                                       "enable_thinking", "?"),
            "ty_le_bi_cat_token": (round(ty_le_bi_cat(), 4)
                                   if "ty_le_bi_cat" in globals() else None),
            "bi_cat_theo_buoc": (bi_cat_theo_buoc()
                                if "bi_cat_theo_buoc" in globals() else None),
            "nap_an_toan": bool(globals().get("NAP_AN_TOAN", False)),
            "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
            "env": run_env(),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'meta')}")
    return stage                      # tên THẬT, có thể khác tên truyền vào


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else "máy cá nhân"
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

In [ ]:
# ═══ Self-test: chạy TRƯỚC khi tốn GPU ═══
# Cell này đỏ thì dừng lại — mọi con số PA/EA sau đó sẽ vô nghĩa.

# (1) Ô cài đặt có thật sự cài được không. Đọc metadata nên nhanh, không phải import.
#     Kiểm ở đây để lỗi pip lộ ra trong 1 giây, thay vì 20 phút nữa lúc nạp model.
from importlib.metadata import version as _ver, PackageNotFoundError as _NoPkg
_goi = {}
for _p in ("vllm", "unsloth", "transformers", "trl", "peft", "torch"):
    try:
        _goi[_p] = _ver(_p)
    except _NoPkg:
        _goi[_p] = None
print("[GÓI] " + " | ".join(f"{k}={v}" for k, v in _goi.items() if v))
# vLLM phải khớp CUDA của torch, nếu không unsloth CHẶN import dù gói vẫn có mặt —
# lúc đó cell nạp model báo "No module named 'vllm'" một cách khó hiểu.
# Wheel khớp CUDA có đuôi "+cuXXX" trong số phiên bản; bản PyPI thì không.
if _goi.get("vllm") and "+cu" not in _goi["vllm"]:
    print("[GÓI] ⚠ vllm=" + _goi["vllm"] + " là bản PyPI (dựng cho CUDA 13). Nếu cell nạp "
          "model báo \"No module named 'vllm'\" thì cài lại bằng wheel khớp CUDA — "
          "dòng WARNING của unsloth in sẵn URL đúng.")

_thieu = [k for k, v in _goi.items() if v is None]
if _thieu:
    raise RuntimeError(
        "Thiếu gói: " + ", ".join(_thieu) + " — ô cài đặt (cell #1) đã thất bại.\n\n"
        "Cách chữa: mở Cửa sổ dòng lệnh (góc dưới trái), chạy\n"
        "    pip install -U unsloth vllm\n"
        "xem lỗi thật, xong Restart session rồi chạy lại TỪ CELL #2 (bỏ qua cell #1).")

# vLLM 0.23 cấm toàn bộ transformers 5.x. Gói nào đó nâng lên 5 thì chặn ngay tại đây,
# đừng để phát hiện sau 4 phút nạp model. (sentence-transformers ≥ 6 là thủ phạm hay gặp.)
if str(_goi["transformers"]).split(".")[0] != "4":
    raise RuntimeError(
        "transformers=" + str(_goi["transformers"]) + " — vLLM 0.23 chỉ chạy với "
        "transformers 4.x, gói nào đó đã nâng nó lên.\n"
        "Chữa: pip install \"transformers==4.57.6\" rồi Restart session.")

# (2) Executor có tái tạo đúng nhãn vàng không.
_ok = sum(dsl.check_ea(dsl.execute_program(s["qa"]["program"], s.get("table") or []),
                       s["qa"].get("exe_ans")) for s in test_all)
print(f"[SELF-TEST] executor tái tạo exe_ans trên test: {_ok}/{len(test_all)}")
assert _ok / len(test_all) > 0.99, "Executor không tái tạo được nhãn vàng — DỪNG."
assert dsl.execute_program("divide(5310, add(1, 0.15))", []) is None   # lồng nhau
assert dsl.check_pa("add(1, 2)", "add(2, 1)")[0]                       # giao hoán
assert dsl.check_ea(0.6066481994, "0.60665")                           # làm tròn 5 chữ số
print("[SELF-TEST] ✅ executor / PA / EA đạt")

## §2. Model

In [ ]:
# ═══════════════ MODEL — Qwen3-8B 4-bit ═══════════════
# Tham số lấy từ reference/original_notebooks/inference_with_difference_models.ipynb:
#   load_in_4bit=True, fast_inference=True, temperature=0.1
# max_tokens thì KHÔNG giữ: nâng 3000 → 8192 vì ở mức cũ 5–10 % mẫu bị cắt giữa lúc
# suy nghĩ, mất trắng. Xem lý do đầy đủ ở ô cấu hình GPU.
MODEL_NAME = "unsloth/Qwen3-8B"
MODEL_TAG  = "Qwen3-8B"

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Không thấy GPU. Runtime → Change runtime type → L4 GPU.")
_GPU, _VRAM = torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1024**3
_CC = torch.cuda.get_device_capability(0)
if _CC[0] < 7:
    raise RuntimeError(f"{_GPU} (CC {_CC[0]}.{_CC[1]}) không chạy được vLLM. "
                       f"Runtime → Change runtime type → A100 GPU.")

# ═══ Tham số ẢNH HƯỞNG KẾT QUẢ — CỐ ĐỊNH trên mọi GPU ═══
# Trước đây max_seq đổi theo GPU (A100 15000 / L4 13500) nên hai máy cho kết quả
# không so thẳng được. Giờ khoá cứng: đổi GPU chỉ đổi tốc độ, không đổi đầu vào.
#
# TRẦN SINH = 4096. Đây là mức ĐO ĐƯỢC là tối ưu, không phải chọn bừa:
#     nấc 2, cùng prompt, cùng GPU, chỉ khác trần —
#       4096 → 30 lượt bị cắt | 28 mẫu mất | EA 0.6479 | 300 mẫu đúng
#       8192 → 30 lượt        | 28 mẫu     | EA 0.6479 | 300 mẫu đúng
#     Gấp đôi ngân sách cứu ĐÚNG 0 mẫu. Số mẫu vượt trần không phụ thuộc trần, nên 4096
#     đã qua điểm bão hoà; 8192 chỉ tốn thêm thời gian. (Dưới 4096 thì mất thêm mẫu.)
#
# ~6 % mẫu vẫn chạm trần — nay KHÔNG bỏ mặc nữa: run_pipeline vớt chúng bằng một lượt
# sinh lại với suy nghĩ TẮT (xem `vot_mau_bi_cat`). Đó mới là cách chữa, không phải trần.
#
# max_seq 17000 theo ngân sách (neo vào phép đo thật bằng tokenizer):
#     prompt bước 2 xấu nhất = 7464 + 4096 = 11560
#     ngân sách              = 17000 − 4096 = 12904   → dư 1344 token
# Ô §3 đo lại bằng tokenizer thật và tự cắt ngữ cảnh + báo động nếu tính sai.
#
# ⚠ ĐỪNG nâng tiếp. Đã có phép so SẠCH: nấc 2 chạy hai lần với CÙNG prompt engineered,
# cùng model, cùng GPU, chỉ khác trần token —
#     trần 4096 → 30 lượt sinh bị cắt | 28 mẫu mất trắng | EA 0.6479 | 300 mẫu đúng
#     trần 8192 → 30 lượt             | 28 mẫu           | EA 0.6479 | 300 mẫu đúng
# Gấp đôi ngân sách cứu được ĐÚNG 0 mẫu, đổi lại ~50 % thời gian (10,9 → 16,4 phút).
#
# Số mẫu vượt ngân sách KHÔNG phụ thuộc ngân sách → những lượt đó thực tế không có điểm
# dừng. Mà chúng cũng không lặp (§4 đo trung vị lặp = 0.0 ở nấc 2), nên repetition_penalty
# cũng không phải thuốc. Coi đây là sàn ~6 %, đều ở mọi nấc: ghi nhận rồi bỏ qua.
TEMPERATURE, MAX_TOKENS = 0.1, 4096
MAX_SEQ_LENGTH = 17000
REPETITION_PENALTY = 1.0

# ═══ Tham số chỉ ảnh hưởng TỐC ĐỘ — chỉnh theo VRAM ═══
if _VRAM < 20:
    raise RuntimeError(
        f"{_GPU} chỉ {_VRAM:.0f} GB — không đủ cho max_seq={MAX_SEQ_LENGTH}.\n"
        f"Hạ max_seq xuống thì kết quả KHÔNG so được với các nấc khác, nên thà dừng "
        f"còn hơn ra một con số không dùng được. Đổi sang L4 hoặc A100.")
# util giữ 0.85 (hạ từ 0.88 sau một lần vLLM không dựng nổi engine vì VRAM còn sót).
# MAX_NUM_SEQS trả về mức cũ được vì max_seq đã từ 25000 xuống 17000, áp lực KV giảm hẳn.
#
# BATCH_SIZE = 512 để 497 mẫu vào ĐÚNG MỘT LÔ. Đo từ log thật: lô 400 mẫu chạy
# 1,88 s/mẫu, lô 97 mẫu còn lại chạy 2,83 s/mẫu — chậm hơn 50 % vì không lấp đầy GPU mà
# vẫn phải đợi mẫu dài nhất. Gộp một lô tiết kiệm ~1,5 phút MỖI lượt sinh; nấc 4 và nấc 5
# có nhiều lượt nên cộng lại đáng kể.
elif _VRAM < 30:                           # L4 24GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.86, 16, 512
elif _VRAM < 60:                           # A100 40GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.85, 48, 512
else:                                      # A100 80GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.85, 128, 512
DTYPE = torch.float16 if _CC[0] < 8 else None

print(f"[GPU] {_GPU} | {_VRAM:.1f} GB | CC {_CC[0]}.{_CC[1]}")
print(f"[CFG] max_seq={MAX_SEQ_LENGTH} max_tokens={MAX_TOKENS} temp={TEMPERATURE} "
      f"(cố định mọi GPU) | batch={BATCH_SIZE} max_num_seqs={MAX_NUM_SEQS} (theo VRAM)")

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import unsloth
from unsloth import FastLanguageModel
from vllm import SamplingParams

torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)

import shutil as _sh

# Đặt True nếu model tải về bị thiếu trọng số: tắt hf_transfer thì tải chậm hơn vài phút
# nhưng có kiểm tra và tải tiếp được. Lưu ý: `export` trong Cửa sổ dòng lệnh KHÔNG tới
# được kernel notebook — phải đặt ở đây.
TAI_CHAM_CHO_CHAC = False
if TAI_CHAM_CHO_CHAC:
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    print("[MODEL] đã tắt hf_transfer — tải chậm hơn nhưng chắc hơn")

_free = _sh.disk_usage("/").free / 1024**3
_t_nap = time.time()
print(f"[MODEL] Đang tải {MODEL_NAME} ... (đĩa trống {_free:.0f} GB)")
if _free < 15:
    print("[MODEL] ⚠ dưới 15 GB trống — model ~15 GB, tải dễ đứt giữa chừng.")
print("[MODEL] ⏳ Mất 4–7 PHÚT. Tải xong rồi vLLM còn dựng CUDA graph — đoạn đó")
print("[MODEL]    KHÔNG có thanh tiến trình, nhìn như treo nhưng không phải.")
print("[MODEL]    Muốn biết còn sống: xem MỐC GIỜ ở các dòng INFO bên dưới. Nó nhích")
print("[MODEL]    lên là đang chạy. Đứng im quá 10 phút mới đáng nghi.")

# enable_prefix_caching: system prompt (~1 800 token) GIỐNG HỆT ở cả 497 request, nên
# vLLM chỉ cần prefill nó một lần rồi dùng lại. Tiết kiệm phần lớn thời gian prefill.
# Không đổi token sinh ra — mỗi request vẫn có seed riêng.
_NAP_KW = dict(model_name     = MODEL_NAME,
               dtype          = DTYPE,
               max_seq_length = MAX_SEQ_LENGTH,
               load_in_4bit   = True,
               fast_inference = True)
try:                                  # bản unsloth cũ không nhận tham số này
    import inspect as _insp
    if "enable_prefix_caching" in _insp.signature(
            FastLanguageModel.from_pretrained).parameters:
        _NAP_KW["enable_prefix_caching"] = True
except Exception:                                    # noqa: BLE001
    pass
NAP_AN_TOAN = False          # True = đã phải lùi về chế độ an toàn, có ghi vào meta

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        **_NAP_KW, gpu_memory_utilization=GPU_MEM_UTIL, max_num_seqs=MAX_NUM_SEQS)
except (RuntimeError, ValueError) as _e:
    # ── vLLM dựng engine hỏng vì CUDA ──
    # KHÔNG phải tải model hỏng: model đã nằm trên đĩa rồi. Lỗi ở bước cấp phát KV cache
    # và dựng CUDA graph — thường do VRAM trống ít hơn lần trước (GPU khác, hoặc tiến
    # trình cũ còn giữ bộ nhớ), khiến số block KV tính ra quá nhỏ.
    if "CUDA error" in str(_e) or "invalid argument" in str(_e):
        print("[MODEL] ⚠ vLLM KHÔNG dựng được engine (CUDA error).")
        print(f"[MODEL]   Đang dùng: max_seq={MAX_SEQ_LENGTH} util={GPU_MEM_UTIL} "
              f"max_num_seqs={MAX_NUM_SEQS}")
        try:
            _free, _tot = torch.cuda.mem_get_info()
            print(f"[MODEL]   VRAM trống: {_free/1024**3:.1f}/{_tot/1024**3:.1f} GB"
                  + ("   ← ĐÃ BỊ CHIẾM. Restart session rồi chạy lại TỪ Ô #2."
                     if _free / _tot < 0.9 else ""))
        except Exception:                                    # noqa: BLE001
            pass
        print("[MODEL]   Thử lại ở CHẾ ĐỘ AN TOÀN: bỏ CUDA graph, hạ VRAM và số chuỗi.")
        print("[MODEL]   Ba thứ đó chỉ đổi TỐC ĐỘ — mỗi request đã có seed riêng nên")
        print("[MODEL]   thành phần lô không ảnh hưởng token sinh ra.")
        gc.collect()
        torch.cuda.empty_cache()
        _an = dict(gpu_memory_utilization=min(GPU_MEM_UTIL, 0.80),
                   max_num_seqs=max(8, MAX_NUM_SEQS // 4))
        try:
            model, tokenizer = FastLanguageModel.from_pretrained(
                **_NAP_KW, enforce_eager=True, **_an)
        except TypeError:                 # bản unsloth không nhận enforce_eager
            model, tokenizer = FastLanguageModel.from_pretrained(**_NAP_KW, **_an)
        GPU_MEM_UTIL = _an["gpu_memory_utilization"]
        MAX_NUM_SEQS = _an["max_num_seqs"]
        NAP_AN_TOAN = True
        print(f"[MODEL] ✅ nạp được ở chế độ an toàn (util={GPU_MEM_UTIL} "
              f"max_num_seqs={MAX_NUM_SEQS}) — chậm hơn, kết quả không đổi.")
    # ── Thiếu trọng số: shard safetensors tải dở còn trong cache ──
    elif "not initialized from checkpoint" in str(_e):
        raise RuntimeError(
            "Model thiếu trọng số — bản tải dở trong cache HuggingFace.\n\n"
            "Bước 1 — xoá cache. Mở Cửa sổ dòng lệnh (góc dưới trái):\n"
            "    rm -rf ~/.cache/huggingface/hub/models--unsloth--Qwen3-8B*\n"
            "    df -h / | tail -1          # kiểm luôn, cần ≥ 20 GB trống\n\n"
            "Bước 2 — đặt TAI_CHAM_CHO_CHAC = True ở ĐẦU CHÍNH Ô NÀY.\n"
            "    (`export` trong terminal không tới được kernel notebook.)\n\n"
            "Bước 3 — Restart session, chạy lại TỪ CELL #2 (bỏ qua ô cài đặt).\n\n"
            "Hỏng y hệt lần nữa thì không phải do tải: khi đó là bản 4-bit của unsloth "
            "không khớp bộ nạp của vLLM, phải đổi phiên bản chứ không phải tải lại.") from _e
    else:
        raise

SAMPLING = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                          repetition_penalty=REPETITION_PENALTY, seed=RANDOM_SEED)
LORA_REQUEST = None          # nấc 3 trở đi có thể gán adapter đã SFT vào đây

from collections import Counter as _Counter
LY_DO_DUNG = _Counter()          # finish_reason: "stop" = tự kết thúc, "length" = BỊ CẮT
LY_DO_THEO_BUOC = {}             # desc → Counter riêng, để tách bước 1 với bước 2


def ty_le_bi_cat():
    """Phần trăm lượt sinh bị cắt vì chạm max_tokens, TÍNH TỪ LẦN reset gần nhất."""
    t = sum(LY_DO_DUNG.values())
    return (LY_DO_DUNG.get("length", 0) / t) if t else 0.0


def bi_cat_theo_buoc():
    """Tỉ lệ bị cắt TÁCH RIÊNG cho bước 1 và bước 2.

    Phải tách vì prompt bước 2 (self-eval ở nấc 4, ACE ở nấc 5) chứa NGUYÊN lời giải
    bước 1, nên dài hơn bước 1 rất nhiều. Bước 2 bị cắt nhiều hơn nghĩa là phương pháp
    bị PHA LOÃNG — mất cơ hội sửa, chứ không phải sửa sai. Con số gộp chung không phân
    biệt được hai chuyện đó.

    Gom theo đuôi của desc ("vòng3/step1" và "vòng7/step1" cùng vào "step1").
    """
    gom = {}
    for k, c in LY_DO_THEO_BUOC.items():
        gom.setdefault(k.rsplit("/", 1)[-1], _Counter()).update(c)
    return {b: {"n": sum(c.values()), "bi_cat": c.get("length", 0),
                "ty_le": round(c.get("length", 0) / max(1, sum(c.values())), 4)}
            for b, c in sorted(gom.items())}


def in_bi_cat_theo_buoc():
    d = bi_cat_theo_buoc()
    if not d:
        return
    # In cả khi chỉ có MỘT bước: nấc 1 và 2 cũng cần biết tỉ lệ chạm trần của mình,
    # nếu không thì mãi tới nấc 4 mới thấy con số đó.
    print("   Bị cắt vì trần token, tách theo bước:")
    for b, v in d.items():
        print(f"     {b:<10}{v['ty_le']:>7.1%}  ({v['bi_cat']}/{v['n']} lượt)")
    if "step2" in d and "step1" in d and d["step2"]["ty_le"] > d["step1"]["ty_le"] + 0.02:
        print("     ⚠ bước 2 bị cắt nhiều hơn bước 1 → hiệu quả của phương pháp đang bị")
        print("       PHA LOÃNG (mất cơ hội sửa). Hiệu số đo được là cận DƯỚI.")


def dat_lai_bo_dem():
    """Gọi ngay trước mỗi nấc. Không gọi thì tỉ lệ là cộng dồn cả phiên — gồm cả
    lượt warmup và (ở nấc 5) toàn bộ pha A, không phản ánh nấc đang đo."""
    LY_DO_DUNG.clear()
    LY_DO_THEO_BUOC.clear()


def generate(prompts, sampling_params=None, desc=None, batch_size=None):
    """Sinh theo lô qua vLLM — như vòng lặp trong notebook cũ."""
    if not prompts:
        return []
    sp = sampling_params or SAMPLING
    bs = batch_size or BATCH_SIZE
    outs, t0 = [], time.time()
    nb = (len(prompts) + bs - 1) // bs
    for i in range(nb):
        chunk = prompts[i*bs:(i+1)*bs]
        kw = {"sampling_params": sp}
        if LORA_REQUEST is not None:
            kw["lora_request"] = LORA_REQUEST
        _res = model.fast_generate(chunk, **kw)
        for o in _res:                      # đếm lý do dừng để biết có bị cắt không
            for _x in o.outputs:            # sinh nhiều mẫu thì đếm CẢ K mẫu
                _r = getattr(_x, "finish_reason", "?")
                LY_DO_DUNG[_r] += 1
                if desc:
                    LY_DO_THEO_BUOC.setdefault(desc, _Counter())[_r] += 1
        # 1 mẫu → trả chuỗi (y như cũ); nhiều mẫu → trả list[str] cho self-consistency.
        outs.extend((o.outputs[0].text if len(o.outputs) == 1
                     else [_x.text for _x in o.outputs]) for o in _res)
        if desc:
            el = time.time() - t0
            print(f"    {desc}: lô {i+1}/{nb} | {el:.0f}s | "
                  f"ETA {el/(i+1)*(nb-i-1):.0f}s", end="\r")
    gc.collect(); torch.cuda.empty_cache()
    if desc:
        _b = LY_DO_THEO_BUOC.get(desc, _Counter())
        _c, _n = _b.get("length", 0), max(1, sum(_b.values()))
        print(f"    {desc}: xong {len(prompts)} prompt trong {time.time()-t0:.0f}s"
              f" | bị cắt vì trần token: {_c/_n:.1%} ({_c} lượt)" + " "*8)
    return outs

_t = torch.cuda.get_device_properties(0).total_memory/1024**3
print(f"[MODEL] ✅ sẵn sàng sau {(time.time()-_t_nap)/60:.1f} phút | "
      f"VRAM {_t - torch.cuda.mem_get_info()[0]/1024**3:.1f}/{_t:.1f} GB")
_ = generate(["xin chào"], SamplingParams(temperature=0, max_tokens=4))
print("[WARMUP] ✅")

## §3. Chọn model nền

In [ ]:
# ═══ Cấu hình ACE — notebook TỰ CHỌN cái chưa có kết quả ═══
# Ba cấu hình, mỗi cấu hình học MỘT playbook riêng nên phải chạy riêng. Nhưng không
# phải sửa công tắc: chạy hết notebook, ô cuối in ra còn cấu hình nào, bấm Ctrl+F10
# từ ô này là chạy tiếp — model và mọi thứ đã nạp vẫn còn trên GPU.
#
#   (ACE chồng lên prompt nào, dùng adapter SFT?, tên nấc sẽ ghi ra)
# Nhánh chạy trên adapter SFT ĐÃ BỎ khỏi lộ trình. Nấc 3 đo được EA 0,6761 so
# với 0,6781 của nấc 2 — McNemar p = 1,000, KTC [-0,036; +0,034], 41 mẫu đúng
# thêm đổi 42 mẫu hỏng đi. Giữ nhánh đó là tốn GPU để đo lại một số 0.
#
CAC_CAU_HINH = [("engineered", False, "05_ace_base"),
                ("basic",      False, "05c_ace_basic_base")]

_con_lai = [(p, u, st) for p, u, st in CAC_CAU_HINH
            if load_stage(st, quiet=True) is None]
assert _con_lai, "Mọi cấu hình ACE đã có kết quả. Sang bước tiếp theo."

_CH = _con_lai[0]
ACE_TREN_PROMPT, USE_SFT_ADAPTER = _CH[0], _CH[1]
print(f"[CẤU HÌNH] chạy {len(CAC_CAU_HINH) - len(_con_lai) + 1}/{len(CAC_CAU_HINH)}: "
      f"'{_CH[2]}' (ACE trên '{_CH[0]}', SFT={_CH[1]}) | còn lại sau lượt này: "
      f"{[x[2] for x in _con_lai[1:]] or 'không còn'}")

LORA_REQUEST = None
BASE_TAG, PREV_STAGE = "base", "04_selfeval_base"
print("[MODEL] Qwen3-8B gốc — nhánh SFT đã bỏ khỏi lộ trình (xem nấc 3)")
print(f"[MODEL] nấc trước để so sánh: {PREV_STAGE}")

In [ ]:
prompt_kit = PromptKit(tokenizer=tokenizer, model_name=MODEL_NAME)
PROMPT_LEVEL = "engineered"
USE_SELFEVAL = True

_think = getattr(prompt_kit, "enable_thinking", None)
print(f"[PROMPT] mức = {PROMPT_LEVEL} | self-eval = {USE_SELFEVAL} | "
      f"suy nghĩ = {'template tự quyết (Qwen3: BẬT)' if _think is None else _think}")
if _think is False:
    print("[PROMPT] ⚠ suy nghĩ đang TẮT — lệch bản tham chiếu, PA sẽ hụt "
          "~10 điểm. Dấu hiệu: 497 mẫu chạy xong trong ~1 phút.")
print(f"         thang lồng nhau: basic={len(prompt_kit.BASIC_SYSTEM_PROMPT)} ký tự"
      f" ⊂ no_fewshot={len(prompt_kit.NO_FEWSHOT_SYSTEM_PROMPT)}"
      f" ⊂ engineered={len(prompt_kit.ENGINEERED_SYSTEM_PROMPT)}"
      f" | self-eval={len(prompt_kit.SELF_EVAL_SYSTEM_PROMPT)}")

# Đo bằng tokenizer THẬT trên 40 mẫu có ngữ cảnh DÀI NHẤT
BUDGET = MAX_SEQ_LENGTH - MAX_TOKENS
_clen = lambda s: (len(" ".join(s.get("pre_text") or [])) +
                   len(" ".join(s.get("post_text") or [])) + len(str(s.get("table") or "")))
_probe = sorted(test_all, key=_clen, reverse=True)[:40]
_bul = "\n".join(["- Khi hỏi tốc độ tăng trưởng, dùng subtract(gia_tri_moi, gia_tri_cu), "
                  "divide(#0, gia_tri_cu)."] * 7)
# Lời giải bước 1 dài nhất có thể là đúng MAX_TOKENS token (model sinh chạm trần).
# Phải đo ở mức đó, không thì bật suy nghĩ vào là prompt bước 2 tràn ngân sách.
_unit = "Phân tích chi tiết từng bước của bảng số liệu. "
_prev = _unit * max(1, MAX_TOKENS // max(1, len(tokenizer(_unit).input_ids)))
_prev += "\n```plaintext\nprogram: divide(1,2)\nanswer: 0.5\n```"

def _measure():
    a = [len(tokenizer(prompt_kit.step1(s, _bul, level=PROMPT_LEVEL)).input_ids)
         for s in _probe]
    b = ([len(tokenizer(prompt_kit.step2(s, _prev, _bul)).input_ids) for s in _probe]
         if USE_SELFEVAL else [0])
    return a, b

_a, _b = _measure()
print(f"[PROMPT] (40 mẫu dài nhất) step1 max={max(_a)} | step2 max={max(_b)} | "
      f"ngân sách={BUDGET}")

if max(max(_a), max(_b)) > BUDGET:
    prompt_kit.max_prev_chars = 3000
    _cap = _clen(_probe[0])
    for _ in range(6):
        _cap = int(_cap * 0.80)
        prompt_kit.max_ctx_chars = max(1200, _cap)
        _a, _b = _measure()
        if max(max(_a), max(_b)) <= BUDGET:
            break
    assert max(max(_a), max(_b)) <= BUDGET, "Không cắt đủ — giảm MAX_TOKENS hoặc dùng GPU lớn hơn."
    _hit = sum(1 for s in test_all if _clen(s) > prompt_kit.max_ctx_chars)
    print(f"[PROMPT] ⚠ đã bật cắt ngữ cảnh (max_ctx_chars={prompt_kit.max_ctx_chars}); "
          f"{_hit}/{len(test_all)} mẫu bị cắt ({_hit/len(test_all)*100:.1f}%)")
    print(f"[PROMPT]   GHI LẠI con số này khi báo cáo.")
else:
    print("[PROMPT] ✅ mọi prompt đều lọt ngân sách, không cần cắt")

## §4. Thành phần ACE

* **Embedder** — `multilingual-e5-base` cho truy hồi ngữ nghĩa tiếng Việt.
* **Retriever** — Tier-1 (bullet đã chứng minh có ích, luôn vào prompt) + Tier-2 (chọn động).
* **QualityGate** — chặn bullet chứa số liệu cụ thể, chứa năm, `multiply(#n,100)`, phép lồng
  nhau, tham chiếu `#N` sai, trùng bullet cũ; và **chạy thử công thức trên bộ số giả**.
* **Reflector** — backend `slm` dùng chính Qwen3, giữ thiết lập *constrained*.

In [ ]:
# ═══ Cấu hình ACE ═══
TOP_K_TIER1, TOP_K_TIER2 = 3, 4
MAX_PLAYBOOK_BULLETS = 30
MAX_BULLETS_PER_CLUSTER = 3
TIER1_MAX, TIER1_MIN_USES, TIER1_MIN_LIFT = 5, 12, 0.01

TRAIN_SUBSET, ACE_ROUND_SIZE, MAX_REFLECT_PER_ROUND = 600, 32, 12
# dev=120 thì 1 mẫu = 0.83 điểm EA — chọn snapshot tốt nhất trên tín hiệu đó là chọn
# theo may rủi. Lượt trước composite dao động 0.588–0.672 giữa các lần đo CÙNG một
# playbook 3 bullet. Gấp đôi dev và đo thưa đi: tổng chi phí không đổi (8×120 ≈ 4×240)
# nhưng mỗi con số đáng tin gấp đôi.
DEV_SUBSET, EVAL_EVERY_ROUNDS = 240, 5
DEV_EVAL_USE_SELFEVAL = False        # dev eval chỉ 1 bước cho nhanh
USE_VERIFY_ITERATE, VERIFY_REQUIRE_PA = True, False
# "openai" = gpt-4o-mini, ĐÚNG như bản ACE gốc (USE_HYBRID_REFLECTOR=True)  ← mặc định
# "slm"    = chính Qwen3-8B đang chạy — không gọi API ngoài, giữ thiết lập constrained
REFLECTOR_BACKEND = "openai"
REFLECTOR_MODEL   = "gpt-4o-mini"
COMPOSITE_EA_W, COMPOSITE_PA_W = 0.60, 0.40

# Reflector = gpt-4o-mini là ĐÚNG thiết kế ACE gốc. Đổi lại, nấc 5 bước ra khỏi thiết
# lập *constrained* mà nấc 1–4 đang theo (không LLM/API ngoài). Hệ quả khi viết bài:
# bảng thang bậc phải GHI RÕ nấc 5 dùng API ngoài, đừng để người đọc tưởng cùng một
# thiết lập. Trường `reflector_backend`/`reflector_model` trong meta giữ dấu vết đó.
#
# Muốn có thêm nhánh constrained để so thì đổi lại "slm" và chạy pha A lần nữa
# (~45 phút GPU). Khi đó tách được "cơ chế ACE có tác dụng" khỏi "Reflector mạnh có
# tác dụng" — phép so đáng giá nhất, và là câu phản biện chắc chắn bị hỏi.
#
# Chi phí API: ~230 lời gọi cho một lượt pha A, khoảng 0,2 USD với gpt-4o-mini.
if REFLECTOR_BACKEND == "openai":
    _kname = "OPENAI_API_KEY"
    if not os.environ.get(_kname):
        try:                                   # lấy từ Colab Secrets
            from google.colab import userdata
            os.environ[_kname] = userdata.get(_kname)
        except Exception:
            pass
    assert os.environ.get(_kname), (
        f"REFLECTOR_BACKEND={REFLECTOR_BACKEND!r} cần {_kname}.\n"
        f"Thêm vào Colab Secrets (biểu tượng chìa khoá ở thanh bên trái), bật "
        f"'Notebook access', rồi chạy lại ô này.")
    print(f"[ACE] ⚠ Reflector gọi API ngoài ({REFLECTOR_BACKEND}) — kết quả thuộc nhóm "
          f"UNCONSTRAINED, báo cáo tách khỏi thang bậc chính.")

embedder = pb_mod.Embedder()
retriever = pb_mod.Retriever(embedder=embedder, k_tier1=TOP_K_TIER1, k_tier2=TOP_K_TIER2,
                             tier1_max=TIER1_MAX, tier1_min_uses=TIER1_MIN_USES,
                             tier1_min_lift=TIER1_MIN_LIFT,
                             max_bullets=MAX_PLAYBOOK_BULLETS)
# DEDUP_THRESH phải khớp embedder. Bản gốc dùng MiniLM/bge (cặp không liên quan ~0.1–0.3)
# nên 0.85 là "trùng" thật. multilingual-e5 nén mọi cặp vào ~0.70–0.90 → 0.85 là SÀN.
# Giữ 0.85 đã khiến 113/217 đề xuất bị loại oan dù playbook chỉ có 1 bullet.
# MIN_OPS: bản gốc ép 2 phép; 64 % tập test ViNumQA là câu MỘT phép nên ép 2 là chặn
# hẳn lời khuyên cho nhóm lớn nhất.
# Lượt trước ghi lại độ tương đồng thật của từng lần loại:
#   0.93→4  0.94→19  0.95→26  0.96→18  0.97→14  0.98→4  0.99→3
# Tức sàn tương đồng của e5 trên câu tiếng Việt cùng chủ đề nằm ở ~0.95, không phải
# 0.85 như MiniLM. Đặt 0.98 để chỉ chặn trùng lặp thật sự.
DEDUP_THRESH, MIN_OPS = 0.98, 1

# ACE_TREN_PROMPT đã được ô #6 chọn tự động từ CAC_CAU_HINH.
#   "engineered" → nấc 5 chuẩn. Prompt này ĐÃ chứa ánh xạ từ khoá → phép toán, tức
#                  đúng loại tri thức ACE định rút ra; ACE chồng lên gần như hết đất.
#   "basic"      → nấc 5c. Khoảng trống rộng hơn hẳn: ACE có TỰ khám phá lại được thứ
#                  người viết prompt đã viết tay không?
if ACE_TREN_PROMPT == "basic":
    ACE_LEVEL, ACE_SELFEVAL = "basic", False
    ACE_STAGE_TIEN_TO, ACE_PREV = "05c_ace_basic", "01_basic"
else:
    ACE_LEVEL, ACE_SELFEVAL = "engineered", True
    ACE_STAGE_TIEN_TO, ACE_PREV = "05_ace", None      # None = giữ PREV_STAGE của cell #6
print(f"[ACE] chồng lên prompt '{ACE_LEVEL}' | self-eval={ACE_SELFEVAL} | "
      f"so với nấc '{ACE_PREV or PREV_STAGE}'")
gate = pb_mod.QualityGate(embedder=embedder, retriever=retriever,
                          dedup_thresh=DEDUP_THRESH, min_ops=MIN_OPS)
print(f"[ACE] cổng: dedup≥{DEDUP_THRESH} | tối thiểu {MIN_OPS} phép DSL | "
      f"≤{MAX_BULLETS_PER_CLUSTER} bullet/cụm")
curator = pb_mod.Curator(gate, max_bullets_per_cluster=MAX_BULLETS_PER_CLUSTER)
# prompt_level PHẢI khớp nấc đang chạy. Reflector được đưa cho xem "model đã được dặn
# sẵn những gì" rồi bị cấm đề xuất lại — mà bản trước luôn đưa prompt `engineered`, kể
# cả khi nấc 5c chạy prompt `basic` (vốn KHÔNG có khối ánh xạ từ khoá nào). Thế là 5c
# bị cấm khám phá đúng thứ nó sinh ra để đo.
refl = refl_mod.Reflector(prompt_kit, pb_mod.all_bullets, backend=REFLECTOR_BACKEND,
                          api_model=REFLECTOR_MODEL, prompt_level=ACE_LEVEL,
                          generate_fn=generate, sampling_params=SAMPLING)

print(f"[ACE] embedding={embedder.name} | top_k={TOP_K_TIER1}+{TOP_K_TIER2} | "
      f"trần={MAX_PLAYBOOK_BULLETS} bullet | reflector={REFLECTOR_BACKEND}")
if not embedder.available:
    print("[ACE] ⚠ không có embedding đa ngữ — truy hồi yếu đi đáng kể.")

# Gọi thử MỘT lời trước khi vào pha A. Reflector nuốt lỗi API (dùng fallback) nên nếu
# key sai thì pha A vẫn chạy 45 phút rồi mới lòi ra playbook rỗng — quá đắt để phát hiện.
if REFLECTOR_BACKEND == "openai":
    _thu = refl._call_openai(['Trả lời đúng một JSON: {"ok": true}'])
    if not _thu or not _thu[0]:
        raise RuntimeError(
            f"Gọi thử {REFLECTOR_BACKEND} KHÔNG trả về gì — xem dòng [REFLECT] ⚠ ngay trên.\n"
            f"Thường là: key sai, hết hạn mức, hoặc tài khoản chưa nạp tiền.\n"
            f"Sửa xong hãy chạy lại ô này. Đừng vào pha A khi bước này chưa qua.")
    print(f"[ACE] ✅ gọi thử {REFLECTOR_MODEL} OK → {str(_thu[0])[:70]}")

## §5. Pha A — học playbook trên train

Mỗi vòng xử lý 32 mẫu, năm bước đều gom lô qua vLLM. Cứ 3 vòng chấm trên dev và **giữ
snapshot tốt nhất** theo `0.6·EA + 0.4·PA` — playbook dùng cho pha B là snapshot đó, không
phải playbook của vòng cuối.

Checkpoint sau mỗi vòng: Colab ngắt giữa chừng thì chạy lại cell là tiếp tục được.

In [ ]:
train_sub = data.stratified_sample(train_all, TRAIN_SUBSET, seed=RANDOM_SEED)
dev_sub = data.stratified_sample(valid_all, DEV_SUBSET, seed=RANDOM_SEED + 1)
print(f"[PHA A] train_sub={len(train_sub)} dev_sub={len(dev_sub)}")
_nz = sum(1 for s in train_sub if data.is_noisy_gold(s))
print(f"[PHA A] nhãn nhiễu trong train_sub: {_nz} ({_nz/len(train_sub)*100:.1f}%) — "
      f"Reflector sẽ nhận tín hiệu sai ở các mẫu này.")
_ = clusters.print_cluster_distribution(train_sub, "train_sub")

In [ ]:
# Cắt tiền tố số nấc bằng dấu "_" đầu tiên, KHÔNG cắt cứng 3 ký tự: "05_ace" có
# tiền tố 3 ký tự nhưng "05c_ace_basic" có 4, cắt cứng ra "_ace_basic" dư dấu gạch
# → tên file thành progress__ace_basic_base.json.
ACE_TAG = f"{ACE_STAGE_TIEN_TO.split(chr(95), 1)[1]}_{BASE_TAG}"
PROGRESS = os.path.join(OUTPUT_DIR, f"progress_{ACE_TAG}.json")
PLAYBOOK_PATH = os.path.join(OUTPUT_DIR, f"playbook_{ACE_TAG}.txt")

trainer = AceTrainer(prompt_kit, generate, retriever, curator, refl,
                     prompt_level=ACE_LEVEL, use_selfeval=ACE_SELFEVAL,
                     sp_step1=SAMPLING, sp_step2=SAMPLING,
                     round_size=ACE_ROUND_SIZE, max_reflect=MAX_REFLECT_PER_ROUND,
                     use_verify=USE_VERIFY_ITERATE, verify_require_pa=VERIFY_REQUIRE_PA,
                     log_path=os.path.join(LOG_DIR, f"ace_history_{ACE_TAG}.jsonl"))

best = {"playbook": None, "score": -1.0, "EA": 0.0, "PA": 0.0, "round": None}
baseline_dev, eval_log, start_round = None, [], 0
n_rounds = (len(train_sub) + ACE_ROUND_SIZE - 1) // ACE_ROUND_SIZE

# Chỉ nối tiếp checkpoint khi CẤU HÌNH Y NGUYÊN. Đổi ngưỡng cổng rồi nối tiếp thì
# vừa vô nghĩa (nửa số bullet học bằng luật cũ) vừa âm thầm — checkpoint đã chạy hết
# sẽ làm vòng lặp rỗng, pha A bị bỏ qua mà không báo gì.
CAU_HINH_ACE = {"dedup_thresh": DEDUP_THRESH, "min_ops": MIN_OPS,
                  "ace_tren_prompt": ACE_TREN_PROMPT, "ace_selfeval": ACE_SELFEVAL,
                  "doi_chung_thoai_hoa": globals().get("DOI_CHUNG_THOAI_HOA"),
                  "ty_le_bi_cat": round(ty_le_bi_cat(), 4),
                "reflector": REFLECTOR_BACKEND, "train_subset": TRAIN_SUBSET,
                "round_size": ACE_ROUND_SIZE, "top_k": [TOP_K_TIER1, TOP_K_TIER2]}

if os.path.exists(PROGRESS):
    _st = json.load(open(PROGRESS, encoding="utf-8"))
    _cu, _noi_tiep = _st.get("cau_hinh"), False
    if _st.get("tag") != ACE_TAG:
        print("[RESUME] checkpoint thuộc tag khác — bỏ qua, học lại từ đầu.")
    elif _cu != CAU_HINH_ACE:
        _bak = PROGRESS + ".cu"
        os.replace(PROGRESS, _bak)
        print("[RESUME] ⚠ CẤU HÌNH ĐÃ ĐỔI so với checkpoint → KHÔNG nối tiếp, học lại từ đầu.")
        print(f"          cũ : {_cu}")
        print(f"          mới: {CAU_HINH_ACE}")
        print(f"          checkpoint cũ giữ lại ở {os.path.basename(_bak)}")
    elif _st["round"] + 1 >= n_rounds:
        print(f"[RESUME] ⚠ pha A ĐÃ CHẠY XONG ở lượt trước ({_st['round']+1}/{n_rounds} vòng)")
        print( "          và cấu hình không đổi → chạy lại sẽ KHÔNG học thêm bullet nào.")
        print(f"          Muốn học lại từ đầu: xoá {PROGRESS}")
        _noi_tiep = True
    else:
        _noi_tiep = True
    if _noi_tiep:
        trainer.playbook = _st["playbook"]; start_round = _st["round"] + 1
        best, baseline_dev, eval_log = _st["best"], _st.get("baseline_dev"), _st["eval_log"]
        retriever.tier1_ids.update(_st.get("tier1_ids", []))
        curator.bullet_cluster.update(_st.get("bullet_cluster", {}))
        for k, v in _st.get("usage", {}).items():
            retriever.usage[k] = v
        trainer.qg_reasons.update(_st.get("qg_reasons", {}))
        trainer.quarantine.update(_st.get("quarantine", []))
        print(f"[RESUME] tiếp từ vòng {start_round}, "
              f"{len(pb_mod.all_bullets(trainer.playbook))} bullet")

def dev_eval(pb, label):
    r = pipeline.run_pipeline(dev_sub, prompt_kit, generate, prompt_level=ACE_LEVEL,
                              use_selfeval=DEV_EVAL_USE_SELFEVAL, playbook=pb,
                              retriever=retriever, sp_step1=SAMPLING, sp_step2=SAMPLING,
                              desc=label, keep_raw=False)
    m = pipeline.summarize(r, label)
    m["composite"] = round(COMPOSITE_EA_W * m["EA"] + COMPOSITE_PA_W * m["PA_strict"], 4)
    return m

print(f"\n{'═'*72}\n  PHA A — {n_rounds} vòng × {ACE_ROUND_SIZE} mẫu\n{'═'*72}")

if start_round == 0:
    # Mốc tham chiếu: playbook rỗng. KHÔNG coi là ứng viên — nếu coi nó là "best" thì
    # khi hoà điểm, playbook rỗng sẽ thắng và mọi bullet học được bị vứt đi.
    _bm = dev_eval(pb_mod.empty_playbook(), "dev/rỗng")
    baseline_dev = {"score": _bm["composite"], "EA": _bm["EA"], "PA": _bm["PA_strict"]}
    eval_log.append({"round": -1, **_bm})
    print(f"  [DEV] mốc playbook rỗng: EA={_bm['EA']:.4f} PA={_bm['PA_strict']:.4f} "
          f"composite={_bm['composite']:.4f}")

_t0 = time.time()
for ri in range(start_round, n_rounds):
    batch = train_sub[ri*ACE_ROUND_SIZE:(ri+1)*ACE_ROUND_SIZE]
    if not batch:
        break
    _rows, info = trainer.run_round(batch, ri)
    print(f"\n── Vòng {ri+1}/{n_rounds} | {len(pb_mod.all_bullets(trainer.playbook))} bullet "
          f"| Tier1={len(retriever.tier1_ids)} ──")
    print(f"   lô này: EA={sum(r['ea'] for r in _rows)/len(_rows):.3f} "
          f"| sai={info['n_fail']} | +{len(info['added'])} | -{len(info['evicted'])}")
    for bid, txt in info["added"]:
        print(f"     + [{bid}] {txt[:78]}...")
    if info["promoted"]:
        print(f"     ↑ Tier-1: {info['promoted']}")

    if (ri+1) % EVAL_EVERY_ROUNDS == 0 or ri == n_rounds-1:
        m = dev_eval(trainer.playbook, f"dev/v{ri+1}")
        m["round"] = ri; m["bullets"] = len(pb_mod.all_bullets(trainer.playbook))
        eval_log.append(m)
        print(f"   [DEV] EA={m['EA']:.4f} PA={m['PA_strict']:.4f} "
              f"composite={m['composite']:.4f} (best={best['score']:.4f})")
        if m["composite"] > best["score"]:
            best = {"playbook": trainer.playbook, "score": m["composite"],
                    "EA": m["EA"], "PA": m["PA_strict"], "round": ri}
            open(PLAYBOOK_PATH, "w", encoding="utf-8").write(trainer.playbook)
            print("   [DEV] ✅ snapshot tốt nhất mới")

    json.dump({"round": ri, "tag": ACE_TAG, "cau_hinh": CAU_HINH_ACE,
               "quarantine": sorted(trainer.quarantine),
               "playbook": trainer.playbook, "best": best,
               "baseline_dev": baseline_dev, "eval_log": eval_log,
               "tier1_ids": sorted(retriever.tier1_ids),
               "bullet_cluster": curator.bullet_cluster,
               "usage": dict(retriever.usage), "qg_reasons": dict(trainer.qg_reasons)},
              open(PROGRESS, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

if best["playbook"] is None:
    best = {"playbook": trainer.playbook, "score": -1.0, "EA": 0, "PA": 0, "round": "cuối"}
PLAYBOOK = best["playbook"]
open(PLAYBOOK_PATH, "w", encoding="utf-8").write(PLAYBOOK)
print(f"\n[PHA A] xong sau {(time.time()-_t0)/60:.1f} phút | "
      f"{len(pb_mod.all_bullets(PLAYBOOK))} bullet | dev EA={best['EA']:.4f}")
if baseline_dev:
    _d = best["score"] - baseline_dev["score"]
    print(f"[PHA A] so với mốc rỗng: composite {_d:+.4f}"
          + ("  ✅" if _d > 0 else "  ⚠ chưa cải thiện trên dev"))

In [ ]:
print(f"{'═'*78}\n  PLAYBOOK HỌC ĐƯỢC\n{'═'*78}")
print(PLAYBOOK)
if trainer.qg_reasons:
    print("\n[QUALITY GATE] lý do bullet bị loại:")
    for k, v in trainer.qg_reasons.most_common(15):
        print(f"  {k:<44}{v:>5}")
    print("\n  Đọc: nhiều 'chua_so_lieu_cu_the' ⇒ Reflector đang chép đáp án;")
    print("       nhiều 'verify_khong_sua_duoc' ⇒ bullet nghe hay nhưng vô dụng.")

## §6. Pha B — chấm trên tập test

In [ ]:
dat_lai_bo_dem()          # tỉ lệ bị cắt tính riêng cho nấc này
STAGE = f"{ACE_STAGE_TIEN_TO}_{BASE_TAG}"
if ACE_PREV:
    PREV_STAGE = ACE_PREV
print(f"\n{'═'*74}\n  NẤC: {STAGE} | {len(pb_mod.all_bullets(PLAYBOOK))} bullet "
      f"| {len(test_all)} mẫu\n{'═'*74}")
_t0 = time.time()
rows = pipeline.run_pipeline(test_all, prompt_kit, generate,
                             prompt_level=ACE_LEVEL, use_selfeval=ACE_SELFEVAL,
                             playbook=PLAYBOOK, retriever=retriever,
                             sp_step1=SAMPLING, sp_step2=SAMPLING, desc=STAGE)
metrics = pipeline.summarize(rows, STAGE)
metrics["minutes"] = round((time.time()-_t0)/60, 1)
pipeline.print_summary(metrics)
in_bi_cat_theo_buoc()


## §7. ACE cộng thêm bao nhiêu

In [ ]:
_prev_rows = load_stage(PREV_STAGE)
if _prev_rows is None:
    print(f"[SO SÁNH] ⚠ chưa có '{PREV_STAGE}' → chạy 04_self_evaluation.ipynb trước.")
else:
    _mp = pipeline.summarize(_prev_rows, PREV_STAGE)
    print(f"\n{'═'*84}\n  NẤC 5 vs {PREV_STAGE} — giá trị của ACE\n{'═'*84}")
    print(f"{'nấc':<24}{'EA':>10}{'PA_strict':>12}{'PA_loose':>11}")
    for _n, _m in [(PREV_STAGE, _mp), (STAGE, metrics)]:
        print(f"{_n:<24}{_m['EA']:>10.4f}{_m['PA_strict']:>12.4f}{_m['PA_loose']:>11.4f}")
    print(f"\n  Δ EA = {metrics['EA']-_mp['EA']:+.4f} | "
          f"Δ PA_strict = {metrics['PA_strict']-_mp['PA_strict']:+.4f}")
    for _k in ("ea", "pa_strict"):
        stats.compare_pair(_prev_rows, rows, key=_k, label=f"nấc 5 so với {PREV_STAGE}",
                           name_base=PREV_STAGE, name_variant=STAGE)
    print(f"\n  Với n={len(test_all)}, chênh lệch dưới ~2 điểm EA thường chưa đạt p<0.05 —")
    print(f"  đó là giới hạn cỡ mẫu, không phải ACE thất bại.")

## §8. Đối chứng: bullet ngẫu nhiên

Câu hỏi mà người phản biện chắc chắn sẽ đặt: *lợi ích đến từ việc chọn đúng bullet, hay chỉ
vì prompt dài thêm?*

Cell dưới chạy lại đúng số lượng bullet nhưng **chọn ngẫu nhiên**. Nếu kết quả ngang ngửa
truy hồi thật thì cơ chế xếp hạng không có giá trị, và có thể bỏ embedding đi cho rẻ.

Tốn thêm ~45 phút — bật khi còn ngân sách.

In [ ]:
RUN_RANDOM_CONTROL = True     # đặt False để bỏ qua (tiết kiệm ~20 phút)

# ⚠ Đối chứng này chỉ có nghĩa khi playbook LỚN HƠN k — nếu không, cả hai bên đều lấy
# TOÀN BỘ bullet và nhận prompt giống hệt nhau. Lượt trước playbook có 3 bullet, k=7:
# hai lần chạy input y hệt, chênh 2.2 điểm EA (đo thật: cùng adapter, chỉ đổi card)
# — đó là NHIỄU, không phải tác dụng của
# truy hồi. Vẫn chạy (số nhiễu đó có giá trị riêng) nhưng phải gọi đúng tên.
_k_th = TOP_K_TIER1 + TOP_K_TIER2
_n_bl = len(pb_mod.all_bullets(PLAYBOOK))
DOI_CHUNG_THOAI_HOA = _n_bl <= _k_th
if DOI_CHUNG_THOAI_HOA:
    print(f"[ĐỐI CHỨNG] ⚠ playbook {_n_bl} bullet ≤ k={_k_th} → hai bên nhận prompt GIỐNG"
          f" HỆT nhau.\n              Kết quả đo được là NHIỄU giữa hai lần chạy, "
          f"KHÔNG phải tác dụng của truy hồi.")
else:
    print(f"[ĐỐI CHỨNG] playbook {_n_bl} bullet > k={_k_th} → truy hồi thật sự có chọn lọc.")

if RUN_RANDOM_CONTROL:
    class RandomRetriever(pb_mod.Retriever):
        """Chọn bullet ngẫu nhiên — nhóm đối chứng cho cơ chế truy hồi."""
        def __init__(self, *a, seed=123, **kw):
            super().__init__(*a, **kw); self._rng = random.Random(seed)
        def retrieve(self, question, pb, k_tier1=None, k_tier2=None, record=False):
            k = ((self.k_tier1 if k_tier1 is None else k_tier1)
                 + (self.k_tier2 if k_tier2 is None else k_tier2))
            bl = pb_mod.all_bullets(pb)
            if not bl or k <= 0:
                return "", []
            ch = self._rng.sample(bl, min(k, len(bl)))
            return "\n".join(f"- {b['content']}" for b in ch), [b["id"] for b in ch]

    dat_lai_bo_dem()          # tỉ lệ bị cắt của riêng nhóm đối chứng
    _rr = RandomRetriever(embedder=embedder, k_tier1=TOP_K_TIER1, k_tier2=TOP_K_TIER2)
    rows_rand = pipeline.run_pipeline(test_all, prompt_kit, generate,
                                      prompt_level=ACE_LEVEL, use_selfeval=ACE_SELFEVAL,
                                      playbook=PLAYBOOK, retriever=_rr,
                                      sp_step1=SAMPLING, sp_step2=SAMPLING, desc="ngau-nhien")
    m_rand = pipeline.summarize(rows_rand, "bullet ngẫu nhiên")
    pipeline.print_summary(m_rand)
    in_bi_cat_theo_buoc()

    _d = metrics["EA"] - m_rand["EA"]
    print(f"\n  truy hồi theo điểm {metrics['EA']:.4f}  vs  ngẫu nhiên {m_rand['EA']:.4f} "
          f"→ Δ = {_d:+.4f}")
    stats.compare_pair(rows_rand, rows, key="ea",
                       label="truy hồi theo điểm so với bullet ngẫu nhiên (cùng số bullet)",
                       name_base="ngau_nhien", name_variant="truy_hoi")
    print(f"\n  → {'Cơ chế truy hồi CÓ giá trị' if _d > 0.01 else 'Truy hồi gần như KHÔNG khác chọn bừa'}")
    # Tên phải mang tiền tố của CẤU HÌNH. `BASE_TAG` là "base" ở CẢ HAI cấu hình
    # (engineered và basic đều không dùng SFT), nên tên cũ `05_ace_random_base`
    # bị cả hai lượt ghi đè lên nhau — lượt `basic` sẽ xoá mất đối chứng của lượt
    # `engineered` sau 110 phút GPU, và cặp ablation ở `07` thành so chéo nền prompt.
    save_stage(f"{ACE_STAGE_TIEN_TO}_random_{BASE_TAG}", rows_rand, m_rand,
               extra={"note": "đối chứng bullet ngẫu nhiên"})

## §9. Bullet nào có công, bullet nào có tội

In [ ]:
if _prev_rows is not None:
    _fix = [(a, b) for a, b in zip(_prev_rows, rows) if b["ea"] and not a["ea"]]
    _brk = [(a, b) for a, b in zip(_prev_rows, rows) if a["ea"] and not b["ea"]]
    print(f"  Sửa đúng thêm: {len(_fix)} | Làm hỏng: {len(_brk)} | "
          f"Thực thu: {len(_fix)-len(_brk):+d}")

    _used = Counter(b for r in rows for b in r["used_bullets"])
    _win = Counter(b for _a, v in _fix for b in v["used_bullets"])
    _lose = Counter(b for _a, v in _brk for b in v["used_bullets"])
    _txt = {b["id"]: b["content"] for b in pb_mod.all_bullets(PLAYBOOK)}

    print(f"\n{'═'*100}\n  ĐÓNG GÓP TỪNG BULLET\n{'═'*100}")
    print(f"  {'bullet':<12}{'sửa':>5}{'hỏng':>6}{'net':>6}{'lượt dùng':>11}  nội dung")
    for bid, n in _used.most_common():
        net = _win[bid] - _lose[bid]
        print(f"  {bid:<12}{_win[bid]:>5}{_lose[bid]:>6}{net:>+6}{n:>11}  "
              f"{_txt.get(bid,'(đã loại)')[:50]}{'  ← nên gỡ' if net < 0 else ''}")

    _bad = [b for b in _used if _win[b] - _lose[b] < 0]
    if _bad:
        print(f"\n  {len(_bad)} bullet hại nhiều hơn lợi: {_bad}")
        print(f"  → Gỡ khỏi {PLAYBOOK_PATH}, rồi chạy lại §6–§7 (không cần học lại pha A).")

    print(f"\n{'═'*84}\n  5 CA ACE SỬA ĐÚNG\n{'═'*84}")
    for a, b in _fix[:5]:
        print(f"\n  [{b['id']}] {b['question'][:84]}")
        print(f"    gold        : {b['gold_program']}  = {b['gold_answer']}")
        print(f"    {PREV_STAGE:<12}: {a['final_program']}  = {a['pred_value']}")
        print(f"    + ACE       : {b['final_program']}  = {b['pred_value']}")
        print(f"    bullet dùng : {', '.join(b['used_bullets']) or '(không)'}")

## §10. Lưu kết quả

In [ ]:
STAGE_DA_GHI = save_stage(STAGE, rows, metrics,
           extra={"prompt_level": ACE_LEVEL, "self_eval": ACE_SELFEVAL, "base": BASE_TAG,
                  "prev_stage": PREV_STAGE, "playbook": PLAYBOOK,
                  "n_bullets": len(pb_mod.all_bullets(PLAYBOOK)),
                  "tier1": sorted(retriever.tier1_ids),
                  "train_subset": TRAIN_SUBSET, "top_k": [TOP_K_TIER1, TOP_K_TIER2],
                  "reflector_backend": REFLECTOR_BACKEND,
                  "reflector_model": globals().get("REFLECTOR_MODEL"),
                  "dedup_thresh": DEDUP_THRESH, "min_ops": MIN_OPS,
                  "ace_tren_prompt": ACE_TREN_PROMPT, "ace_selfeval": ACE_SELFEVAL,
                  "doi_chung_thoai_hoa": globals().get("DOI_CHUNG_THOAI_HOA"),
                  "ty_le_bi_cat": round(ty_le_bi_cat(), 4),
                  "embed_model": embedder.name, "eval_log": eval_log,
                  "qg_rejections": dict(trainer.qg_reasons)})
print(f"\n  EA = {metrics['EA']:.4f} | PA_strict = {metrics['PA_strict']:.4f}")
print("  → Chạy 06_final_report.ipynb để có bảng tổng hợp cả 5 nấc.")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  KHỐI ĐỂ GỬI ĐI ĐỐI CHIẾU — bôi đen từ dòng ─── tới hết rồi copy         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
print("\n" + "─" * 74)
print(json.dumps(json.load(open(stage_path(STAGE_DA_GHI, "meta"), encoding="utf-8")),
                 ensure_ascii=False, indent=1))
print("─" * 74)


# ── Còn cấu hình ACE nào chưa chạy? ──
_sau = [st for p, u, st in CAC_CAU_HINH if load_stage(st, quiet=True) is None]
if _sau:
    print(f"{chr(10)}{'═'*74}")
    print(f"  CÒN {len(_sau)} CẤU HÌNH ACE: {_sau}")
    print("  Bấm vào ô #6 rồi Ctrl+F10 — model, embedder, retriever vẫn còn trên GPU.")
    print("  Mỗi cấu hình học playbook RIÊNG nên phải chạy riêng, không gộp được.")
    print(f"{'═'*74}")
else:
    print(f"{chr(10)}✅ Mọi cấu hình ACE đã xong.")


## Kết luận nấc 5

Ba con số cần mang sang báo cáo:

1. **Δ so với nấc 4** kèm p-value — đóng góp thật của ACE.
2. **Δ so với bullet ngẫu nhiên** — chứng minh lợi ích đến từ truy hồi chứ không phải
   prompt dài thêm.
3. **Số bullet và token thêm vào mỗi prompt** — cái giá phải trả.

Nếu ACE ≈ nấc 4, khả năng cao là **chồng lấn với self-eval**: cả hai đều sửa lỗi lúc suy
luận, và self-eval đã sửa hết phần dễ. Khi đó nên thử chạy ACE **không kèm self-eval**
(đặt `use_selfeval=False` ở §6) và so với nấc 2 — nếu ACE một mình thắng self-eval một mình
thì nó là lựa chọn rẻ hơn (một lượt sinh thay vì hai).